# Notebook 09 — Mass Spectrometry Proteomics

**Module:** 10 — Transcriptomics and Proteomics  
**Tier:** 2 — Working competence  
**Notebook:** 09 of 12  
**Time estimate:** 80 minutes

> RNA tells you what might be made. Proteins tell you what is actually doing work.
> Mass spectrometry is the dominant technology for protein identification
> and quantification at proteome scale.

---
## Step 1 — Motivation

mRNA levels correlate only moderately with protein levels (~0.4–0.6 Pearson r).
Post-translational modifications, protein stability, and translational regulation
decouple mRNA from protein abundance. For many biological questions — especially
drug targets, enzymatic activity, structural function — proteins are the relevant
measurement.

---
## Step 2 — Intuition

**LC-MS/MS workflow:**
1. **Digest:** proteins are digested with trypsin (cuts at K/R) → peptides
2. **Separate:** liquid chromatography (LC) separates peptides by hydrophobicity
3. **Ionize:** electrospray ionization (ESI) converts eluting peptides to gas-phase ions
4. **MS1 (survey scan):** measure masses of intact peptide ions (precursor m/z)
5. **MS2 (fragmentation):** select precursor, fragment by collision, measure fragment masses
6. **Database search:** match fragment spectra against in-silico predicted spectra
   from a protein FASTA database → peptide sequence identification
7. **Protein inference:** roll up peptide identifications to protein-level

**Output:** A protein × sample intensity matrix — analogous to the gene × sample
count matrix in RNA-seq.

---
## Step 3 — Biological Background

**Trypsin digestion:**  
Trypsin cleaves at the C-terminal side of lysine (K) and arginine (R), except when
followed by proline (P). This is highly specific and reproducible, generating peptides
of 6–20 amino acids — ideal for mass spectrometry.

**Peptide spectrum match (PSM):**  
Each MS2 scan is matched to a peptide sequence. The score is the similarity between
observed fragment m/z values and theoretical b/y ions for a candidate peptide.

**Target-decoy FDR:**  
The protein FASTA database is reversed (decoy). Both target and decoy are searched.
$$\text{FDR} \approx \frac{\text{decoy hits}}{\text{target hits}}$$
PSMs are filtered at 1% FDR — meaning ~1% of reported matches are false positives.

**Dynamic range challenge:**  
The proteome spans ~7–8 orders of magnitude in abundance (albumin at mg/mL vs.
transcription factors at fg/mL). Mass spectrometers typically cover 4–5 orders of
magnitude in a single run — deep coverage requires fractionation.

**Missing values:**  
Low-abundance proteins are not detected in every sample. This creates a systematic
missing-not-at-random pattern (proteins missing in low-intensity samples). Missing
value imputation is a key pre-processing challenge.

---
## Step 4 — Mathematical Explanation

**Peptide m/z:**
$$m/z = \frac{M + z \cdot 1.00728}{z}$$

where $M$ = peptide monoisotopic mass, $z$ = charge state, $1.00728$ = proton mass.

**b/y ion series:**  
For peptide $aa_1 aa_2 \ldots aa_n$:
- b-ion $k$: fragment containing residues 1 to $k$ (N-terminal fragment)
  $m(b_k) = \sum_{i=1}^{k} m(aa_i) + 1.00728$ (for z=1)
- y-ion $k$: fragment containing residues $n-k+1$ to $n$ (C-terminal fragment)
  $m(y_k) = \sum_{i=n-k+1}^{n} m(aa_i) + 18.0106 + 1.00728$ (water + proton)

**Target-decoy FDR estimation:**  
Sort PSMs by score (descending). At each score threshold $s$:
$$\text{FDR}(s) = \frac{D(s)}{T(s)}$$
where $D(s)$ = decoy hits with score $\geq s$, $T(s)$ = target hits with score $\geq s$.

In [ ]:
# Step 6 — Python: Simulate proteomics data — trypsin digestion, PSMs, FDR
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# ---- Amino acid residue masses (monoisotopic) ----
AA_MASSES = {
    'A': 71.037, 'R': 156.101, 'N': 114.043, 'D': 115.027,
    'C': 103.009, 'E': 129.043, 'Q': 128.059, 'G': 57.021,
    'H': 137.059, 'I': 113.084, 'L': 113.084, 'K': 128.095,
    'M': 131.040, 'F': 147.068, 'P': 97.053,  'S': 87.032,
    'T': 101.048, 'W': 186.079, 'Y': 163.063, 'V': 99.068,
}
PROTON = 1.007276
WATER  = 18.010565

def peptide_mass(sequence):
    """Monoisotopic mass of peptide."""
    return sum(AA_MASSES.get(aa, 0) for aa in sequence) + WATER

def peptide_mz(sequence, charge=2):
    return (peptide_mass(sequence) + charge * PROTON) / charge

def trypsin_digest(protein_seq, missed_cleavages=0):
    """Trypsin digest: cleave at K/R, not before P."""
    peptides = []
    start = 0
    n = len(protein_seq)
    cleavage_sites = []
    for i in range(n - 1):
        if protein_seq[i] in ('K', 'R') and protein_seq[i + 1] != 'P':
            cleavage_sites.append(i + 1)
    cleavage_sites.append(n)  # C-terminus

    # 0 missed cleavages
    prev = 0
    for site in cleavage_sites:
        peptides.append(protein_seq[prev:site])
        prev = site
    return [p for p in peptides if len(p) >= 5]  # filter very short

# Example protein sequence
example_protein = 'MTEYKLVVVGAGGVGKSALTIQLIQNHFVDEYDPTIEDSYNSAVRHLTKNEKKVLRQWNSQTPGKEFHQALATLREVNHAVQRYKNLRDLHQVAVKDSEDIKEKLDELEARIIDREKARQEAKDDGLDSENVMTQRLSSRKLEKEMEKFRSQLQQLEKRRQQLMRQFRQELERKLESEMEQKLISEEDL'
peptides = trypsin_digest(example_protein)
print(f'Protein length: {len(example_protein)} aa')
print(f'Tryptic peptides: {len(peptides)}')
print('First 5 peptides and their m/z (z=2):')
for pep in peptides[:5]:
    print(f'  {pep:25s}  mass={peptide_mass(pep):.3f}  m/z(z=2)={peptide_mz(pep, 2):.3f}')

# ---- Simulate target-decoy FDR ----
N_PSMS = 1000
# Real PSMs: good scores from a normal distribution
target_scores = rng.normal(3.5, 1.0, N_PSMS)
# Decoy PSMs: random matches, lower scores
decoy_scores  = rng.normal(1.5, 1.2, N_PSMS)

def compute_fdr_curve(target_scores, decoy_scores):
    """Compute FDR at each score threshold."""
    all_scores = np.concatenate([target_scores, decoy_scores])
    is_decoy    = np.concatenate([np.zeros(len(target_scores)),
                                   np.ones(len(decoy_scores))])
    order = np.argsort(all_scores)[::-1]
    all_scores_s = all_scores[order]
    is_decoy_s   = is_decoy[order]

    fdr_vals = []
    n_targets = 0
    n_decoys  = 0
    for i in range(len(all_scores_s)):
        if is_decoy_s[i]:
            n_decoys += 1
        else:
            n_targets += 1
        fdr = n_decoys / max(n_targets, 1)
        fdr_vals.append((all_scores_s[i], fdr, n_targets))
    return fdr_vals

fdr_curve = compute_fdr_curve(target_scores, decoy_scores)
# Find threshold at 1% FDR
threshold_1pct = None
for score, fdr, n_t in fdr_curve:
    if fdr <= 0.01:
        threshold_1pct = score
        n_at_1pct = n_t
print(f'\n1% FDR score threshold: {threshold_1pct:.2f}')
print(f'Proteins identified at 1% FDR: {n_at_1pct}')

In [ ]:
# Simulate and visualize MaxQuant-like proteinGroups output
N_PROTEINS = 500
N_SAMPLES = 6  # 3 control, 3 treatment

# LFQ intensities: log-normal with some treatment effect
baseline = rng.normal(25, 2, N_PROTEINS)  # log2 baseline intensity
treated_effect = np.zeros(N_PROTEINS)
DE_IDX = rng.choice(N_PROTEINS, 50, replace=False)  # 50 changed proteins
treated_effect[DE_IDX] = rng.normal(0, 2.0, 50)  # up or down

protein_intensities = np.zeros((N_PROTEINS, N_SAMPLES))
for j in range(N_SAMPLES):
    noise = rng.normal(0, 0.5, N_PROTEINS)
    if j < 3:  # control
        protein_intensities[:, j] = baseline + noise
    else:       # treatment
        protein_intensities[:, j] = baseline + treated_effect + noise

# Introduce missing values (~20% missing at random + low-abundance correlated)
missing_mask = rng.random((N_PROTEINS, N_SAMPLES)) < 0.20
protein_intensities_missing = protein_intensities.copy()
protein_intensities_missing[missing_mask] = np.nan

pct_missing = np.isnan(protein_intensities_missing).mean() * 100
print(f'Missing values: {pct_missing:.1f}%')

# ---- Visualization ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: b/y ion diagram (conceptual)
ax = axes[0]
peptide_ex = 'LVESGR'
bions = []
yions = []
bmass = PROTON  # b-ion starts with just the proton
for aa in peptide_ex:
    bmass += AA_MASSES.get(aa, 0)
    bions.append(bmass)
ymass = PROTON + WATER
for aa in reversed(peptide_ex):
    ymass += AA_MASSES.get(aa, 0)
    yions.append(ymass)
for k, (b, y) in enumerate(zip(bions, yions[::-1])):
    ax.barh(k - 0.15, b, 0.25, color='steelblue', alpha=0.7, label='b-ions' if k==0 else '')
    ax.barh(k + 0.15, y, 0.25, color='tomato', alpha=0.7, label='y-ions' if k==0 else '')
ax.set_xlabel('m/z (z=1)')
ax.set_title(f'A. b/y ions for "{peptide_ex}"')
ax.set_yticks(range(len(peptide_ex)))
ax.set_yticklabels([f'pos {k+1}' for k in range(len(peptide_ex))])
ax.legend(fontsize=8)

# Panel B: Score distributions target vs. decoy
ax = axes[1]
ax.hist(target_scores, bins=40, alpha=0.6, label='Target (real)', color='steelblue', density=True)
ax.hist(decoy_scores, bins=40, alpha=0.6, label='Decoy (random)', color='tomato', density=True)
if threshold_1pct:
    ax.axvline(threshold_1pct, color='black', ls='--', label=f'1% FDR threshold')
ax.set_xlabel('PSM score')
ax.set_ylabel('Density')
ax.set_title('B. Target vs. decoy score distributions\n(basis for FDR control)')
ax.legend(fontsize=8)

# Panel C: Missing value pattern
ax = axes[2]
sort_by_missing = np.argsort(np.isnan(protein_intensities_missing).mean(axis=1))[::-1]
im = ax.imshow(np.isnan(protein_intensities_missing[sort_by_missing[:80], :]).astype(int),
               aspect='auto', cmap='Reds', interpolation='none')
ax.set_xlabel('Sample')
ax.set_ylabel('Protein (sorted by missingness)')
ax.set_xticks(range(N_SAMPLES))
ax.set_xticklabels(['C1','C2','C3','T1','T2','T3'])
ax.set_title(f'C. Missing value pattern\n({pct_missing:.0f}% missing total)')
plt.colorbar(im, ax=ax, label='Missing (1=yes)')

plt.suptitle('Module 10 NB09: Mass Spectrometry Proteomics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('proteomics_ms.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Compute the m/z of peptide 'ALELFR' at charge states z=1 and z=2.
2. Implement `trypsin_digest()` from scratch and test it on the example protein.
   Verify that all cleavage sites are at K/R (not before P).
3. What is the 1% FDR score threshold in the simulated data? How many PSMs pass it?
4. If 20% of protein intensities are missing, name two biologically reasonable
   imputation strategies and their appropriate use cases.

---
## Step 10 — Quiz

1. What does LC-MS/MS stand for and what does each step do?
2. What is a PSM and what does "1% FDR" mean for PSMs?
3. Why does trypsin cut at K/R and not before P?
4. What is the difference between b-ions and y-ions in MS2?
5. Why is the missing value problem in proteomics different from random missing data?

---
## Step 12 — Reflection

> *[In one paragraph: explain why the mRNA-to-protein correlation is imperfect,
> naming at least three molecular mechanisms that decouple mRNA abundance
> from protein abundance.]*

---
*Next: `10_protein_quantification_methods.ipynb`*